# Differentiable programming — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sigmoid(z): return 1/(1+np.exp(-z))
def bce(p,y):
    p=np.clip(np.asarray(p,dtype=float),1e-9,1-1e-9)
    y=np.asarray(y,dtype=float)
    return -(y*np.log(p)+(1-y)*np.log(1-p))
print('setup ok')

## Treating a program parameter as something gradients can tune
Differentiable programming writes computation as ordinary code but chooses smooth operations so parameters can be adjusted by gradients. These five examples use a soft threshold program, show its loss surface, compute the gradient, run gradient descent, and compare smooth learning with a hard non-differentiable branch.

In [ ]:
# 1) a soft threshold program y=sigmoid(k*(x-t)) produces smooth outputs
x=np.array([0.,1.,2.,3.]); y=np.array([0.,0.,1.,1.]); k=4.; t=1.5; pred=sigmoid(k*(x-t)); loss=float(bce(pred,y).mean())
plt.figure(figsize=(6,3)); plt.plot(x,pred,'-o',label='soft program'); plt.scatter(x,y,c='k',label='labels'); plt.ylim(-.05,1.05); plt.legend(); plt.title(f'soft threshold, loss={loss:.4f}')
assert abs(loss-0.06470184809035148)<1e-12

In [ ]:
# 2) the threshold t has a smooth loss curve with a visible minimum
Ts=np.linspace(0,3,121); losses=np.array([bce(sigmoid(k*(x-T)),y).mean() for T in Ts]); bestT=float(Ts[np.argmin(losses)])
plt.figure(figsize=(6,3)); plt.plot(Ts,losses); plt.axvline(bestT,c='r',ls='--'); plt.xlabel('threshold t'); plt.ylabel('BCE'); plt.title(f'loss surface over program parameter, best t={bestT:.2f}')
assert abs(bestT-1.5)<1e-12

In [ ]:
# 3) finite-difference gradient tells which way to move t
T0=1.0; eps=1e-4; grad=float((bce(sigmoid(k*(x-(T0+eps))),y).mean()-bce(sigmoid(k*(x-(T0-eps))),y).mean())/(2*eps))
plt.figure(figsize=(6,3)); plt.plot(Ts,losses); plt.scatter([T0],[bce(sigmoid(k*(x-T0)),y).mean()],c='r'); plt.arrow(T0,0.5,0.35,0,width=0.01,color='r'); plt.title(f'at t=1.0, gradient={grad:.4f}')
assert grad<0 and abs(grad+0.49966464986062054)<1e-6

In [ ]:
# 4) gradient descent tunes the program parameter
T=1.0; hist=[]
for _ in range(30):
    g=(bce(sigmoid(k*(x-(T+eps))),y).mean()-bce(sigmoid(k*(x-(T-eps))),y).mean())/(2*eps)
    hist.append((T,float(bce(sigmoid(k*(x-T)),y).mean()))); T-=0.2*g
hist=np.array(hist)
plt.figure(figsize=(6,3)); plt.plot(hist[:,0],hist[:,1],'-o',ms=3); plt.xlabel('t during descent'); plt.ylabel('loss'); plt.title(f'descent moves t toward {T:.3f}')
assert abs(T-1.4984574318643302)<1e-9

In [ ]:
# 5) hard branches hide gradients, smooth branches expose them
hard_losses=np.array([np.mean(((x>T).astype(float)-y)**2) for T in Ts]); soft_losses=np.array([bce(sigmoid(k*(x-T)),y).mean() for T in Ts])
plt.figure(figsize=(6,3)); plt.plot(Ts,hard_losses,label='hard step loss'); plt.plot(Ts,soft_losses,label='soft differentiable loss'); plt.legend(); plt.xlabel('t'); plt.title('smooth relaxation gives a usable slope')
assert len(np.unique(hard_losses))<len(np.unique(np.round(soft_losses,6)))